# Portfolio Forecasting and Optimization

This notebook summarizes the end-to-end workflow for the portfolio optimization challenge: loading market data, exploring asset behavior, fitting forecasting models, optimizing a portfolio, and backtesting the strategy.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8")

from src.preprocessing import load_prices, compute_returns
from src.forecasting import run_forecasting
from src.portfolio import optimize_portfolio
from src.backtest import run_backtest

prices = load_prices(raw_dir="../data/raw")
prices.head()

In [ ]:
prices.describe().T

In [ ]:
returns = compute_returns(prices)
returns.describe().T

In [ ]:
normalized = prices / prices.iloc[0] * 100
normalized.plot(figsize=(12, 6), linewidth=1.3)
plt.title("Normalized Asset Prices")
plt.xlabel("Date")
plt.ylabel("Index (start = 100)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
forecast_result = run_forecasting(prices, ticker="TSLA")
print("ARIMA order:", forecast_result["arima_order"])
print("ARIMA metrics:", forecast_result["arima_metrics"])
print("Sequence metrics:", forecast_result["sequence_metrics"])

In [ ]:
tsla_expected_return = float(forecast_result["series"].pct_change().dropna().mean() * 252)
portfolio_result = optimize_portfolio(prices, tsla_expected_return=tsla_expected_return)
print(portfolio_result["weights_df"])
print("Expected return:", portfolio_result["expected_return"])
print("Volatility:", portfolio_result["volatility"])
print("Sharpe ratio:", portfolio_result["sharpe"])

In [ ]:
backtest_result = run_backtest(prices[["TSLA", "SPY", "BND"]], portfolio_result["weights"])
print(backtest_result["strategy_metrics"])
print(backtest_result["benchmark_metrics"])

## Key Takeaways

- The workflow successfully loads and prepares the asset data.
- The sequence-based model produced stronger test metrics than ARIMA for this dataset.
- The optimized portfolio places meaningful weight on TSLA and SPY while keeping BND minimal.
- The backtest suggests the strategy outperformed the static 60/40 benchmark on total return in this sample.